# FRTB – Commodity Curvature Risk Capital (CMCV)

This notebook calculates the **Commodity Curvature Risk Capital Requirement** under the
**Fundamental Review of the Trading Book (FRTB)** Standardised Approach, as defined in
**CRR3 / Basel IV** (Articles 325g, 325p, 325at, 325au, 325ay).

## Workflow Overview
The calculation follows these numbered steps (matching the regulatory text):

| Step | Description |
|------|-------------|
| 3 | Record gross Curvature Value-at-Risk (CVR) positions per instrument |
| 4 | Net positions that share the same risk factor (commodity type) |
| 5–6 | Derive intra- and cross-bucket curvature correlations from delta correlations |
| 8 | Compute bucket-level capital charges (K_b) for upward and downward scenarios |
| 9 | Compute bucket sums (S_b) required for the cross-bucket aggregation |
| 10 | Aggregate across buckets to obtain the Risk-Class Curvature Requirement (RCCR) |
| 11 | Repeat for all three correlation scenarios (Low / Medium / High) and take the maximum |

---

In [ ]:
import pandas as pd
import numpy as np

## Steps 3 & 4 – Gross and Net CVR Positions

**Curvature Value-at-Risk (CVR)** measures the sensitivity of a position to a parallel
shift in the entire risk-factor curve.  Two CVR figures are produced per position:

- **CVR+** – upward shock scenario (prices rise)
- **CVR-** – downward shock scenario (prices fall)

Positions that reference the **same underlying risk factor** must be netted before
aggregation (Article 325g(1)).  In this example the two *Sugar* trades share the same
risk factor, so their CVRs are summed together.

In [ ]:
# ---------------------------------------------------------------------------
# Step 3 – Gross positions as reported (one row per trade/instrument)
# Bucket numbering follows the FRTB commodity bucket taxonomy (Article 325at).
#   Bucket  2 = Energy – Oil/Gas
#   Bucket 10 = Softs / Agricultural
# ---------------------------------------------------------------------------
gross_positions_data = {
    'Bucket':  [10,     10,     2,       2    ],
    'Product': ['Sugar','Sugar','Brent', 'WTI'],
    'CVR+':    [26177,  91620,  124433, -35460],   # upward shock P&L
    'CVR-':    [14197,  49690, -152185,  63068],   # downward shock P&L
}
gross_df = pd.DataFrame(gross_positions_data)

print("--- Step 3: Gross Positions ---")
print(gross_df.to_string(index=False))
print("\n" + "=" * 55 + "\n")

# ---------------------------------------------------------------------------
# Step 4 – Net positions: group by (Bucket, Product) and sum CVRs.
# 'Sugar' appears twice; both entries share the same risk factor, so they
# are combined into a single net row.
# ---------------------------------------------------------------------------
net_df = gross_df.groupby(['Bucket', 'Product'], sort=False).sum().reset_index()

print("--- Step 4: Net Positions ---")
print("(The two Sugar rows have been netted into one.)")
print(net_df.to_string(index=False))
print("\n" + "=" * 55 + "\n")

## Steps 5 & 6 – Correlation Parameters (Medium Scenario)

Under FRTB, curvature correlations are derived directly from the corresponding
**delta correlations** by squaring them (Article 325ay(5)):

$$\rho^{\text{curv}}_{kl} = \left(\rho^{\delta}_{kl}\right)^2$$

Two types of correlation are needed:

| Correlation | Regulatory reference | Applies to |
|-------------|----------------------|------------|
| **Intra-bucket** (ρ_kl) | Art. 325at(2A) | Two different commodities in the *same* bucket |
| **Cross-bucket** (γ_bc) | Art. 325au(a)  | Two different buckets |

The intra-bucket delta correlation for same-commodity type is **95 %**; the
cross-bucket delta correlation between Bucket 2 and Bucket 10 is **20 %**.

In [ ]:
# ---------------------------------------------------------------------------
# Intra-bucket delta correlation – Bucket 2 (Brent vs WTI)
#   Component breakdown (Article 325at(2A)):
#     - Commodity type factor:  95%  (same commodity type, e.g. crude oil)
#     - Tenor factor:          100%  (same tenor)
#     - Basis factor:          100%  (ignored for curvature per Art. 325p(4))
# ---------------------------------------------------------------------------
delta_rho_kl = 0.95 * 1.0 * 1.0     # = 0.95  (95%)

# Cross-bucket delta correlation – Bucket 2 vs Bucket 10 (Article 325au(a))
delta_gamma_bc = 0.20                # = 20%

# Curvature correlations are the square of the corresponding delta correlations
# (Article 325ay(5))
curvature_rho_kl_medium   = delta_rho_kl   ** 2   # intra-bucket
curvature_gamma_bc_medium = delta_gamma_bc ** 2   # cross-bucket

print("--- Steps 5 & 6: Correlation Parameters (Medium Scenario) ---")
print(f"  Intra-bucket delta correlation  ρ_kl       : {delta_rho_kl:.2%}")
print(f"  Intra-bucket curvature corr.    ρ_kl²      : {curvature_rho_kl_medium:.2%}")
print()
print(f"  Cross-bucket delta correlation  γ_bc       : {delta_gamma_bc:.2%}")
print(f"  Cross-bucket curvature corr.    γ_bc²      : {curvature_gamma_bc_medium:.2%}")
print("\n" + "=" * 55 + "\n")

## Step 8 – Bucket-Level Capital Charge (K_b)

For each bucket the capital charge is computed separately for the *upward* (K_b+)
and *downward* (K_b−) scenarios using:

$$K_b^{\pm} = \sqrt{\max\!\left(0,\; \sum_k \max(CVR_k^{\pm},0)^2
             + \sum_{k \ne l} \rho_{kl}^{\text{curv}}\, CVR_k^{\pm}\, CVR_l^{\pm}\,
             \psi(CVR_k^{\pm}, CVR_l^{\pm})\right)}$$

The **ψ (psi) safeguard function** prevents correlation diversification benefits
when *both* CVRs are negative:

$$\psi(x, y) = \begin{cases} 0 & \text{if } x < 0 \text{ and } y < 0 \\ 1 & \text{otherwise} \end{cases}$$

The final K_b for the bucket is `max(K_b+, K_b−)`; the *selected scenario*
(upward or downward) is carried forward to Step 9.

In [ ]:
def calculate_bucket_capital(cvr_data: pd.DataFrame, intra_bucket_corr: float):
    """
    Calculate the bucket-level curvature capital charge.

    Parameters
    ----------
    cvr_data : DataFrame
        Net CVR positions for a single bucket.  Must contain columns
        'CVR+' (upward scenario) and 'CVR-' (downward scenario).
    intra_bucket_corr : float
        Curvature intra-bucket correlation (ρ_kl²) for this bucket.

    Returns
    -------
    kb_plus : float
        Upward-scenario bucket capital (K_b+).
    kb_minus : float
        Downward-scenario bucket capital (K_b−).
    kb_final : float
        Final bucket capital = max(K_b+, K_b−).
    scenario : str
        The dominant scenario: 'Upward' or 'Downward'.
    """

    def psi(x: float, y: float) -> int:
        """Return 0 when both values are negative, otherwise 1.
        Prevents offsetting negative CVRs from reducing the capital charge."""
        return 0 if (x < 0 and y < 0) else 1

    cvr_plus  = cvr_data['CVR+'].values
    cvr_minus = cvr_data['CVR-'].values

    # ---- Upward scenario (K_b+) ----------------------------------------
    # Sum of squared *positive* CVR+ values (negative CVRs contribute nothing)
    sum_sq_plus = np.sum(np.maximum(cvr_plus, 0) ** 2)

    # Cross-term: only applicable when there are at least two risk factors
    # (formula simplified here for the two-factor case)
    corr_term_plus = 0.0
    if len(cvr_plus) > 1:
        corr_term_plus = (
            2 * intra_bucket_corr
            * cvr_plus[0] * cvr_plus[1]
            * psi(cvr_plus[0], cvr_plus[1])
        )

    kb_plus = np.sqrt(max(0.0, sum_sq_plus + corr_term_plus))

    # ---- Downward scenario (K_b−) --------------------------------------
    sum_sq_minus = np.sum(np.maximum(cvr_minus, 0) ** 2)

    corr_term_minus = 0.0
    if len(cvr_minus) > 1:
        corr_term_minus = (
            2 * intra_bucket_corr
            * cvr_minus[0] * cvr_minus[1]
            * psi(cvr_minus[0], cvr_minus[1])
        )

    kb_minus = np.sqrt(max(0.0, sum_sq_minus + corr_term_minus))

    # ---- Select the dominant scenario ----------------------------------
    if kb_plus > kb_minus:
        return kb_plus, kb_minus, kb_plus, "Upward"
    elif kb_minus > kb_plus:
        return kb_plus, kb_minus, kb_minus, "Downward"
    else:
        # Tie-break: choose the scenario with the larger sum of CVRs
        scenario = "Upward" if np.sum(cvr_plus) >= np.sum(cvr_minus) else "Downward"
        return kb_plus, kb_minus, kb_plus, scenario


print("--- Step 8: Bucket-Level Capital (Medium Scenario) ---")

# Bucket 10 – Sugar (single risk factor → no intra-bucket correlation term)
b10_data = net_df[net_df['Bucket'] == 10]
kb10_plus, kb10_minus, kb10_final, scenario10 = calculate_bucket_capital(b10_data, intra_bucket_corr=0.0)
print("Bucket 10 (Sugar) – single risk factor:")
print(f"  K_b+          = {kb10_plus:>10,.0f}")
print(f"  K_b−          = {kb10_minus:>10,.0f}")
print(f"  Final K_b     = {kb10_final:>10,.0f}  →  {scenario10} scenario selected")
print()

# Bucket 2 – Brent & WTI (two risk factors → apply intra-bucket correlation)
b2_data = net_df[net_df['Bucket'] == 2]
kb2_plus, kb2_minus, kb2_final, scenario2 = calculate_bucket_capital(b2_data, intra_bucket_corr=curvature_rho_kl_medium)
print("Bucket 2 (Brent & WTI) – two risk factors, ρ_kl² = {:.2%}:".format(curvature_rho_kl_medium))
print(f"  K_b+          = {kb2_plus:>10,.0f}")
print(f"  K_b−          = {kb2_minus:>10,.0f}")
print(f"  Final K_b     = {kb2_final:>10,.0f}  →  {scenario2} scenario selected")
print("\n" + "=" * 55 + "\n")

## Step 9 – Bucket Sums (S_b)

The bucket sum S_b is the simple algebraic sum of all CVRs within the bucket,
using the CVR column that corresponds to the **selected scenario** (upward or downward)
from Step 8.  S_b is used as the *signed* quantity in the cross-bucket aggregation
to capture diversification benefits.

In [ ]:
def calculate_bucket_sum(cvr_data: pd.DataFrame, scenario: str) -> float:
    """
    Return the bucket sum S_b for the scenario selected in Step 8.

    Parameters
    ----------
    cvr_data : DataFrame
        Net CVR positions for a single bucket.
    scenario : str
        The scenario chosen in Step 8: 'Upward' or 'Downward'.

    Returns
    -------
    float
        Algebraic sum of CVR+ (upward) or CVR- (downward) for the bucket.
    """
    col = 'CVR+' if scenario == 'Upward' else 'CVR-'
    return float(cvr_data[col].sum())


s10 = calculate_bucket_sum(b10_data, scenario10)
s2  = calculate_bucket_sum(b2_data,  scenario2)

print("--- Step 9: Bucket Sums (Medium Scenario) ---")
print(f"  S_b  Bucket 10 (Sugar)     : {s10:>10,.0f}  [{scenario10} scenario]")
print(f"  S_b  Bucket  2 (Brent+WTI) : {s2:>10,.0f}  [{scenario2} scenario]")
print("\n" + "=" * 55 + "\n")

## Step 10 – Cross-Bucket Aggregation (RCCR)

The **Risk-Class Curvature Requirement (RCCR)** aggregates bucket capitals across
buckets using the cross-bucket curvature correlation γ_bc²:

$$\text{RCCR} = \sqrt{\max\!\left(0,\;
  \sum_b K_b^2 + \sum_{b \ne c} \gamma_{bc}^{\text{curv}}\, S_b\, S_c\,
  \psi(S_b, S_c)\right)}$$

The same ψ safeguard function is applied to bucket sums to ensure no
diversification benefit is granted when both bucket sums are negative.

In [ ]:
def calculate_rccr(
    bucket_capitals: list,
    bucket_sums: list,
    cross_bucket_corr: float,
) -> float:
    """
    Aggregate bucket capitals to produce the Risk-Class Curvature Requirement.

    Parameters
    ----------
    bucket_capitals : list of float
        Final K_b values (one per bucket).
    bucket_sums : list of float
        S_b values for each bucket (signed, from the selected scenario).
    cross_bucket_corr : float
        Curvature cross-bucket correlation (γ_bc²).

    Returns
    -------
    float
        RCCR – the total commodity curvature capital charge.
    """

    def psi(x: float, y: float) -> int:
        """Safeguard: return 0 if both sums are negative, else 1."""
        return 0 if (x < 0 and y < 0) else 1

    # Diagonal term: sum of squared bucket capitals
    sum_kb_sq = np.sum(np.array(bucket_capitals) ** 2)

    # Off-diagonal term: cross-bucket diversification (simplified for 2 buckets)
    corr_term = 0.0
    if len(bucket_sums) > 1:
        corr_term = (
            2 * cross_bucket_corr
            * bucket_sums[0] * bucket_sums[1]
            * psi(bucket_sums[0], bucket_sums[1])
        )

    return float(np.sqrt(max(0.0, sum_kb_sq + corr_term)))


rccr_medium = calculate_rccr(
    bucket_capitals=[kb10_final, kb2_final],
    bucket_sums=[s10, s2],
    cross_bucket_corr=curvature_gamma_bc_medium,
)

print("--- Step 10: Cross-Bucket Capital (Medium Scenario) ---")
print(f"  RCCR (Medium Scenario) = £{rccr_medium:,.0f}")
print("\n" + "=" * 55 + "\n")

## Step 11 – All Correlation Scenarios & Final Capital

FRTB requires the calculation to be repeated under three correlation scenarios
(Article 325bb):

| Scenario | Intra-bucket (ρ_kl) | Cross-bucket (γ_bc) |
|----------|--------------------|-----------------|
| **High** | min(1.25 ρ, 1)    | min(1.25 γ, 1)  |
| **Medium** | ρ (base)         | γ (base)         |
| **Low** | max(2ρ−1, 0.75ρ)  | max(2γ−1, 0.75γ) |

Curvature correlations are derived by squaring the corresponding delta
correlations in each scenario.  The **final capital requirement** is the
**maximum RCCR across the three scenarios**.

In [ ]:
# Define the three correlation scenarios with their scaling rules
scenarios = {
    "High":   {
        # Correlations are stressed upward (capped at 100%)
        "rho_delta":   min(delta_rho_kl   * 1.25, 1.0),
        "gamma_delta": min(delta_gamma_bc * 1.25, 1.0),
    },
    "Low":    {
        # Correlations are stressed downward (floored at 75% of base)
        "rho_delta":   max(2 * delta_rho_kl   - 1, 0.75 * delta_rho_kl),
        "gamma_delta": max(2 * delta_gamma_bc - 1, 0.75 * delta_gamma_bc),
    },
    "Medium": {
        # Base (unstressed) correlations
        "rho_delta":   delta_rho_kl,
        "gamma_delta": delta_gamma_bc,
    },
}

results = []

for name, params in scenarios.items():
    # Convert delta correlations to curvature correlations (square them)
    rho_curv   = params["rho_delta"]   ** 2
    gamma_curv = params["gamma_delta"] ** 2

    # Recalculate Bucket 2 with the scenario-specific intra-bucket correlation.
    # Bucket 10 has only one risk factor, so its capital is scenario-independent.
    _, _, kb2_scen, scen2_scen = calculate_bucket_capital(b2_data, intra_bucket_corr=rho_curv)

    # Bucket sum for Bucket 2 depends on which scenario (up/down) was selected above
    s2_scen = calculate_bucket_sum(b2_data, scen2_scen)

    # Full RCCR for this correlation scenario
    rccr_scen = calculate_rccr(
        bucket_capitals=[kb10_final, kb2_scen],
        bucket_sums=[s10, s2_scen],
        cross_bucket_corr=gamma_curv,
    )

    results.append({
        "Scenario":                    name,
        "Intra-Bucket Curv. Corr.": rho_curv,
        "Cross-Bucket Curv. Corr.": gamma_curv,
        "Bucket 2 Capital (K_2)":   kb2_scen,
        "Total Capital (RCCR)":     rccr_scen,
    })

# Sort rows Low → Medium → High for readability
scenario_order = {"Low": 0, "Medium": 1, "High": 2}
results_df = (
    pd.DataFrame(results)
    .sort_values("Scenario", key=lambda x: x.map(scenario_order))
    .set_index("Scenario")
)

print("--- Step 11: Summary Across All Correlation Scenarios ---")
print(results_df.to_string(formatters={
    'Intra-Bucket Curv. Corr.': '{:.2%}'.format,
    'Cross-Bucket Curv. Corr.': '{:.2%}'.format,
    'Bucket 2 Capital (K_2)':   '£{:,.0f}'.format,
    'Total Capital (RCCR)':     '£{:,.0f}'.format,
}))

# The final capital requirement is the maximum across all three scenarios
final_capital = results_df['Total Capital (RCCR)'].max()

print("\n" + "-" * 55)
print(f"  Final Commodity Curvature Capital Requirement: £{final_capital:,.0f}")
print("-" * 55)